In [ ]:

# =================================================================
# DEFINITIONS OF THE MAPS BETWEEN LIE ALGEBRA AND LIE GROUP
# =================================================================
# 1) exp: so(3) -> SO(3) 
# 2) Exp:   R3  -> SO(3)
# 3) log: SO(3) -> so(3)
# 4) Log: SO(3) -> R3
# 5) Operator wedge: R3 -> so(3)
# 6) Operator Vee: so(3) -> R3
# Note that R3 is isomorphic to the Lie Algebra so(3)
# =================================================================

# ===================================
# Wedge operator: R3 -> so(3) 
# ====================================
def CartesianSpace2LieAlgebra_Wedge(vetor):
    # We receive the vector in R3
    x,y,z = vetor.flatten()
    return np.array([[ 0,  -z,   y],
                     [ z,   0,  -x],
                     [-y,  x,   0]])

# ====================================
# Vee operator: so(3) -> R3 
# ====================================
def LieAlgebra2CartesianSpace_Vee(X):
    # We receive a skew symmetric Matrix in the Lie Algebra
    return np.array([-X[1,2],X[0,2],-X[0,1]]).reshape(3,1)

# ====================================
# exp map: so(3) -> SO(3)
# ====================================
def exp_map_from_so3_to_SO3(X):
    theta = np.sqrt(0.5 * np.trace(X.T @ X))

    if theta < 1e-4:
        # Taylor Series Aproximation
        return np.eye(3) + X + (X @ X)/2
    else:
        # Rodrigues Formula for computing the matrix exponential of a skew-symmetric Matrix
        return np.eye(3) + (np.sin(theta) / theta) * X + ((1 - np.cos(theta)) / theta**2) * (X @ X)

# ====================================
# log map: SO(3) -> so(3)
# ====================================
def log_map_from_SO3_to_so3(X):
    # Force the outputs with clip function, bewtween -1 and 1
    cos_theta = np.clip((np.trace(X) - 1.0) / 2.0, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    
    if theta < 1e-4:
        # Taylor Series Aproximation again
        return (X - X.T)/2
    else:
        return (0.5 * theta / np.sin(theta)) * (X - X.T)

# In the case of the Capitalized maps, we need to do a composition, basically
# ====================================
# Exp map: R3 -> SO(3)
# ====================================

def Exp_map_from_R3_to_SO3(vector):
    X = CartesianSpace2LieAlgebra_Wedge(vector)
    return exp_map_from_so3_to_SO3(X)

# ====================================
# Log map: SO(3) -> R3
# ====================================

def Log_map_from_SO3_to_R3(M):
    m = log_map_from_SO3_to_so3(M)
    return LieAlgebra2CartesianSpace_Vee(m)

# ====================================
# Geodesic Distance
# ====================================

# Given two points in the manifold, we compute the distance between them
def Geodesic_distance_SO3(R1, R2):
    # Receives two rotation matrices and computes the geodesic distance between them
    R_new = log_map_from_SO3_to_so3(R1.T @ R2)
    
    # Frobenius norm
    return np.sqrt(np.trace(R_new.T @ R_new))

# ====================================
# Karcher Mean
# ====================================

# Here we consider the scenario of a weighted Karcher Mean, meaning that
# each matrix can have a weight different from 1
def Karcher_weighted_mean(List_of_weights, List_Rot_Matrix):

    # Maximum number of iterations
    Number_iter_max = 10

    # The first estimate is assigned to the matrix associated with the largest weight
    M_previous = List_Rot_Matrix[np.argmax(List_of_weights)]
    norm = 10**10
    eps = 10**(-5)
    ite = 0
    
    # Check the weights
    List_of_weights = List_of_weights/np.sum(List_of_weights)
    
    while ((ite < Number_iter_max) and (norm > eps)):

        # Initialize the skew-symmetric matrix
        M_sum = np.zeros((3,3))

        # ===========================
        # Lifting 
        # ===========================
        # We lift all rotation matrices to the Lie algebra using the log map,
        # and then we calculate the mean in the Lie algebra (because it is a vector space)
        for l in range(len(List_Rot_Matrix)):
            M_sum = M_sum + List_of_weights[l] * log_map_from_SO3_to_so3(M_previous.T @ List_Rot_Matrix[l])

        # ===========================
        # Reproject
        # ===========================
        # We reproject the mean back onto the manifold using the exp map
        M_final = M_previous @ exp_map_from_so3_to_SO3(M_sum)

        # ===========================
        # Difference between the estimates
        # ===========================
        norm = Geodesic_distance_SO3(M_previous, M_final)
        M_previous = M_final
        ite = ite + 1

    # Riemannian centroid calculated
    return M_final